In [ ]:
import numpy as np
import cv2
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

In [ ]:
# select device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
# download variational autoencoder
!wget http://agentspace.org/download/mnist_cvae_069_encoder.pth
!wget http://agentspace.org/download/mnist_cvae_069_decoder.pth

In [ ]:
# load variational autoencoder
encoder = torch.load('mnist_cvae_069_encoder.pth', weights_only=False, map_location=device)
decoder = torch.load('mnist_cvae_069_decoder.pth', weights_only=False, map_location=device)

In [ ]:
encoder.eval()

In [ ]:
decoder.eval()

In [ ]:
# get datasets
train_dataset = datasets.MNIST(root='data', train=True, download=True, transform=transforms.ToTensor())
test_dataset = datasets.MNIST(root='data', train=False, download=True, transform=transforms.ToTensor())

In [ ]:
# select a subset of classes
classes_to_keep = [0, 6, 9]
class_to_id = {class_: i for i, class_ in enumerate(classes_to_keep)}
print(class_to_id)
id_to_class = {i: class_ for class_, i in class_to_id.items()}
print(id_to_class)

In [ ]:
class RemappedSubset(Dataset):
    def __init__(self, original_dataset, indices, class_to_new):
        self.original = original_dataset
        self.indices = indices
        self.class_to_new = class_to_new

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        x, y = self.original[real_idx]
        new_y = self.class_to_new[int(y)]
        return x, new_y

In [ ]:
# create subset
train_indices = [i for i, (_, label) in enumerate(train_dataset) if label in classes_to_keep] # indices for train subset
test_indices = [i for i, (_, label) in enumerate(test_dataset) if label in classes_to_keep] # indices for test subset
train_subset = RemappedSubset(train_dataset, train_indices, class_to_id)
test_subset = RemappedSubset(test_dataset, test_indices, class_to_id)

In [ ]:
# test subset dataset
labels = torch.tensor([label for _, label in train_subset])
print(torch.unique(labels))
labels = torch.tensor([label for _, label in test_subset])
print(torch.unique(labels))

In [ ]:
# get few samples
x_samples = torch.stack([ test_subset[i][0] for i in [0,1,2,3,4,5,6,7,8,9] ])
print(x_samples.shape)

In [ ]:
# encode and decode
y_samples = decoder(encoder(x_samples.to(device)))

In [ ]:
# show the comparison
def show_comparison(x, y):
    plt.figure(figsize=(20, 4))
    for i in range(10):
        input_img = (x_samples[i].squeeze(0).detach().cpu().numpy()*255).astype(np.uint8)
        plt.subplot(2, 10, i+1)
        plt.imshow(input_img, cmap='gray')
        plt.axis('off')
        output_img = (y_samples[i].squeeze(0).detach().cpu().numpy()*255).astype(np.uint8)
        plt.subplot(2, 10, i+1+10)
        plt.imshow(output_img, cmap='gray')
        plt.axis('off')
    plt.show()

show_comparison(x_samples, y_samples)

In [ ]:
# visualize the latent space
n=20
norm = torch.distributions.Normal(0, 1)
grid_x = 3*norm.icdf(torch.linspace(0.05, 0.95, n-1))
grid_y = 3*norm.icdf(torch.linspace(0.05, 0.95, n-1))

def visualize():
    figure = np.zeros((28 * (n-1), 28 * (n-1)))
    with torch.no_grad():
        for i, yi in enumerate(grid_x):
            for j, xi in enumerate(grid_y):
                z = torch.tensor([[xi, yi]]).float().to('cuda')
                x_decoded = decoder(z)
                digit = x_decoded[0].reshape(28, 28).cpu().numpy()
                figure[i * 28: (i + 1) * 28, j * 28: (j + 1) * 28] = digit
    plt.figure(figsize=(6, 6))
    plt.imshow(figure, cmap='Greys_r')
    plt.axis('off')

visualize()
plt.show()

In [ ]:
# manually select representants of 0, 6, 9
representants = [(16,11), (15,6), (8,6)] # (row, col)

In [ ]:
# check and get values
vecs = []
visualize()
colors = ['lightblue', 'orange', 'red']
for k, representant in enumerate(representants):
    i, j = representant
    color = colors[k]
    plt.arrow(10*28, 10*28, 28*(j-10)+14, 28*(i-10)+14, head_width=5, head_length=0, fc=color, ec=color)
    vecs.append((grid_y[j].item(),grid_x[i].item()))
plt.show()

In [ ]:
vecs = torch.tensor(vecs).float().to(device)
vecs

In [ ]:
digits = decoder(vecs).detach().cpu().reshape(-1,28,28).numpy()
plt.imshow(np.concatenate(digits, axis=1), cmap='Greys_r')
plt.axis('off')
plt.show()

In [ ]:
def plane_to_sphere(xy, r=5.0, R=1.0):
    x, y = xy[:,0], xy[:,1]
    xmin, xmax, ymin, ymax = -r, r, -r, r
    theta = 2 * math.pi * (x - xmin) / (xmax - xmin)   # [0, 2π]
    phi   = math.pi * (y - ymin) / (ymax - ymin)       # [0, π]
    X = R * torch.sin(phi) * torch.cos(theta)
    Y = R * torch.sin(phi) * torch.sin(theta)
    Z = R * torch.cos(phi)
    return torch.stack([X, Y, Z], dim=1)

In [ ]:
def sphere_to_plane(XYZ, r=5.0, R=1.0):
    X, Y, Z = XYZ[:,0], XYZ[:,1], XYZ[:,2]
    xmin, xmax, ymin, ymax = -r, r, -r, r
    theta = torch.atan2(Y, X) % (2 * math.pi)
    phi = torch.acos(torch.clamp(Z / R, -1.0, 1.0))
    x = xmin + theta / (2 * math.pi) * (xmax - xmin)
    y = ymin + phi / math.pi * (ymax - ymin)
    return torch.stack([x, y], dim=1)

In [ ]:
vecs

In [ ]:
plane_to_sphere(vecs)

In [ ]:
sphere_to_plane(plane_to_sphere(vecs))

In [ ]:
keys = plane_to_sphere(vecs)
keys

In [ ]:
values = keys[[0,2,1]].clone()
values

In [ ]:
# attention
def att(queries, keys, values, scale=1.0):
    coeffs = torch.softmax(queries @ keys.t() / scale,dim=1)
    print(coeffs)
    return coeffs @ values

In [ ]:
query = keys[1].unsqueeze(0)

In [ ]:
plt.imshow(decoder(sphere_to_plane(query)).detach().cpu().reshape(28,28).numpy(), cmap='Greys_r')
plt.axis('off')
plt.show()

In [ ]:
query

In [ ]:
query@ keys.t()

In [ ]:
torch.softmax(query @ keys.t() / 0.01, dim=1)

In [ ]:
torch.softmax(query @ keys.t() / 0.01, dim=1) @ values

In [ ]:
plt.imshow(decoder(sphere_to_plane(torch.softmax(query @ keys.t() / 0.01, dim=1) @ values)).detach().cpu().reshape(28,28).numpy(), cmap='Greys_r')
plt.axis('off')
plt.show()

In [ ]:
print(values[1])
att(query, keys, values, scale=0.01)

In [ ]:
# encode, attend and decode
y_samples = decoder(sphere_to_plane(att(plane_to_sphere(encoder(x_samples.to(device))),keys,values,scale=0.01)))

In [ ]:
show_comparison(x_samples, y_samples)